In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from torch.nn.functional import pad
import os

In [13]:
os.chdir("/u/sebono/conversational_dominance/information_exchange_labelling")
print(os.getcwd())

from perplexity_labelling import compute_p1, compute_p2, compute_p3
from utils import (
    # VISUALIZATION
    rolling_kde_heatmap_with_turns,
    display_turns_colored_by_kde,
    compute_graph_perplexity,
    correlation_heatmap,
    # HELPERS
    compute_dominance_per_spk,
    expand_multiple_ppl_by_token,
    remove_match_prefix_ppl,
    assign_words_to_bins,
    compute_significance,
    display_colored_sentences
)

/u/sebono/conversational_dominance/information_exchange_labelling


In [14]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from transformers import LlamaForCausalLM, LlamaTokenizerFast

import argparse
import pickle
import os
import json
import pandas as pd

#model_id = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
model_id="gpt2-large"
device = "cuda:1"
perplexity_func = "p1"

if "gpt2" in model_id:
    #model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
    tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
    start_of_sentence=" "
if "Llama" in model_id:
    #model = LlamaForCausalLM.from_pretrained(model_id, device_map="auto")
    tokenizer = LlamaTokenizerFast.from_pretrained(model_id)
    start_of_sentence='<|begin_of_text|>'

In [15]:
import pandas as pd
import re
import numpy as np

multisimo_df_with_annotations = pd.read_csv("/u/sebono/conversational_dominance/data/processed/multisimo/transcript_dominance.csv")
multisimo_df_with_annotations

,file_name,speaker_1_1,speaker_1_2,speaker_1_3,speaker_1_4,speaker_1_5,speaker_2_1,speaker_2_2,speaker_2_3,speaker_2_4,speaker_2_5,speaker_1_dom_score,speaker_2_dom_score,file_content,speaker_1,speaker_2
0,S02,3.0,2.0,2.0,2.0,1.0,4.0,3.0,3.0,4.0,3.0,2.0,3.4,<MOD>: Ok so thanks for coming today. we're go...,P006,P007
1,S03,2.0,2.0,2.0,2.0,2.0,4.0,3.0,3.0,5.0,2.0,2.0,3.4,<MOD> Ok hello we are going to play a quiz. <...,P008,P009
2,S04,2.0,1.0,2.0,4.0,1.0,1.0,1.0,1.0,4.0,1.0,2.0,1.6,"<MOD>: ok Hello, thank you for coming today, w...",P010,P011
3,S05,4.0,4.0,4.0,4.0,3.0,3.0,2.0,3.0,1.0,1.0,3.8,2.0,"<MOD>: Ok hi welcome, thank you for coming tod...",P012,P013
4,S07,4.0,2.0,3.0,4.0,4.0,4.0,2.0,3.0,5.0,3.0,3.4,3.4,"<MOD>: <MOD>: Ok. So welcome, thank you very m...",P016,P017
5,S08,4.0,2.0,3.0,2.0,1.0,3.0,3.0,3.0,3.0,2.0,2.4,2.8,<MOD>: So good evening <SPK2>: Good evening. <...,P018,P019
6,S09,4.0,3.0,3.0,5.0,3.0,4.0,3.0,3.0,5.0,3.0,3.6,3.6,<MOD>: Ok <SPK2>: Yeah so <MOD>: So I would li...,P020,P021
7,S10,3.0,2.0,3.0,4.0,2.0,3.0,2.0,2.0,4.0,2.0,2.8,2.6,<MOD>: Ok. So I would like us to play a quiz. ...,P022,P023
8,S11,5.0,2.0,3.0,3.0,5.0,2.0,1.0,2.0,1.0,1.0,3.6,1.4,"<SPK1>: <MOD>: Right. So, I would like us to p...",P024,P025
9,S13,3.0,1.0,2.0,4.0,2.0,2.0,1.0,2.0,1.0,1.0,2.4,1.4,"<SPK1>: yeah <MOD>: Well hello, thanks very mu...",P028,P029


In [45]:
from glob import glob
import os, re, pickle
import numpy as np
import re
import pandas as pd
from tqdm import tqdm

BASE = "/u/sebono/conversational_dominance/notebooks/information_exchange_labelling/dataset_perplexity_results"


def _pair_distinguishable(spk_dict, alpha=0.05, adjust_p="fdr_bh"):
    """
    Returns True if <SPK1> and <SPK2> are statistically distinguishable
    by any test in compute_significance (after correction if requested).
    """
    # Only consider main speakers that are present
    spks = [s for s in spk_dict.keys() if s.startswith("<SPK")]
    if not ("<SPK1>" in spks and "<SPK2>" in spks):
        return False

    # Your function (must be in scope)
    results_list, results_dict = compute_significance(
        original_ppls_p_spk=spk_dict,
        spk_tokens=["<SPK1>", "<SPK2>"],
        alternative_mw="two-sided",
        equal_var_ttest=False,
        adjust_p=adjust_p
    )

    # Prefer adjusted p if present; otherwise raw p
    best_p = 1.0
    for row in results_list:
        if (row.get("spk_i"), row.get("spk_j")) in [("<SPK1>", "<SPK2>"), ("<SPK2>", "<SPK1>")]:
            p = row.get("p_value_adj", row.get("p_value"))
            if p is not None and not np.isnan(p):
                best_p = min(best_p, p)
    return best_p < alpha

def compute_dominance(dataset, model, ppls, base=BASE, agg=np.nanmean):
    out = {}
    rows = []

    for idx, name in enumerate(list(multisimo_df_with_annotations["file_name"])):
        el = multisimo_df_with_annotations[multisimo_df_with_annotations['file_name']==name]["file_content"]
        dialog = multisimo_df_with_annotations[multisimo_df_with_annotations['file_name']==name]["file_content"][list(el.keys())[0]]

        pattern = r'<(?:SPK[0-9]|MOD)>'
        dialog_lines = re.sub(r"[\[\(].*?[\]\)]", "", dialog).replace("<", "\n<").split("\n")[1:]
        matches = [f"{match}" for match in re.findall(pattern, f"{start_of_sentence}".join(dialog_lines))]
        token_list = [tokenizer(token, return_tensors="pt").input_ids[0] for token in dialog_lines]
        encodings = tokenizer(f"{start_of_sentence}".join(dialog_lines), return_tensors="pt")
        assert np.cumsum([len(token) for token in token_list])[-1] == encodings.input_ids[0].shape
        assert len(matches) == len(dialog_lines)

        all_data = pd.DataFrame({})
        for p_name in ['p1','p2','p3']:
            file_name_sample = f'/u/sebono/conversational_dominance/notebooks/information_exchange_labelling/dataset_perplexity_results/multisimo_{p_name}_{model}/dominance_scores_{name}.pkl'
            with open(file_name_sample, 'rb') as f:
                all_data[f"{p_name}"] = pickle.load(f)[name]

        filtered_tokens_p1, filtered_ppl_p1, filtered_encodings_p1, filtered_matches_p1 = remove_match_prefix_ppl(token_list, all_data['p1'], matches, tokenizer, min_len=3)
        assert np.cumsum([len(token) for token in filtered_tokens_p1])[-1], len(filtered_ppl)

        filtered_tokens_p2, filtered_ppl_p2, filtered_encodings_p2, filtered_matches_p2 = remove_match_prefix_ppl(token_list, all_data['p2'], matches, tokenizer, min_len=3)
        assert np.cumsum([len(token) for token in filtered_tokens_p2])[-1], len(filtered_ppl)

        filtered_tokens_p3, filtered_ppl_p3, filtered_encodings_p3, filtered_matches_p3 = remove_match_prefix_ppl(token_list, all_data['p3'], matches, tokenizer, min_len=3)
        assert np.cumsum([len(token) for token in filtered_tokens_p3])[-1], len(filtered_ppl)

        #assert len(filtered_tokens_p3) == len(token_list)
        assert len(filtered_tokens_p1) == len(filtered_tokens_p2)
        assert len(filtered_tokens_p1) == len(filtered_tokens_p3)
        assert filtered_encodings_p1 == filtered_encodings_p2
        assert filtered_encodings_p1 == filtered_encodings_p3
        filtered_encodings = filtered_encodings_p1

        all_data = pd.DataFrame({"p1":filtered_ppl_p1,"p2":filtered_ppl_p2, "p3":filtered_ppl_p3})
        all_data.bfill(inplace=True)

        baselined_ppls_p1_spk = compute_dominance_per_spk(filtered_ppl_p1, filtered_tokens_p1, filtered_matches_p1, tokenizer)
        baselined_ppls_p2_spk = compute_dominance_per_spk(filtered_ppl_p2, filtered_tokens_p2, filtered_matches_p2, tokenizer)
        baselined_ppls_p3_spk = compute_dominance_per_spk(filtered_ppl_p3, filtered_tokens_p3, filtered_matches_p3, tokenizer)

        out[name] = {}
        out[name]['p1'] = dict(zip(baselined_ppls_p1_spk.keys(), [np.mean(baselined_ppls_p1_spk[key]) for key in baselined_ppls_p1_spk.keys()]))
        out[name]['p2'] = dict(zip(baselined_ppls_p2_spk.keys(), [np.mean(baselined_ppls_p2_spk[key]) for key in baselined_ppls_p2_spk.keys()]))
        out[name]['p3'] = dict(zip(baselined_ppls_p3_spk.keys(), [np.mean(baselined_ppls_p3_spk[key]) for key in baselined_ppls_p3_spk.keys()]))

        # Collect tidy rows
        for ppl_key in ('p1','p2','p3'):
            for spk, val in out[name][ppl_key].items():
                rows.append((name, ppl_key, spk, float(val) if val is not None else np.nan))

    df = pd.DataFrame(rows, columns=["file_name", "ppl", "speaker", "mean_dominance"])

    # Pivot ppl into columns
    pivot = (
        df.pivot_table(index=["file_name", "speaker"],
                       columns="ppl", values="mean_dominance", aggfunc="mean")
        .reindex(columns=["p1","p2","p3"])
        .reset_index()
    )

    # Add human dominance scores
    dom_long = (
        multisimo_df_with_annotations[["file_name", "speaker_1_dom_score", "speaker_2_dom_score"]]
        .melt(id_vars="file_name", var_name="which", value_name="dom")
        .assign(speaker=lambda d: d["which"].map({
            "speaker_1_dom_score": "<SPK1>",
            "speaker_2_dom_score": "<SPK2>",
        }))
        .drop(columns="which")
    )

    result = (
        pivot.merge(dom_long, on=["file_name","speaker"], how="left")
             .loc[:, ["file_name","speaker","dom","p1","p2","p3"]]
             .sort_values(["file_name","speaker"])
             .reset_index(drop=True)
    )

    return result, out

In [46]:
DATASET = "multisimo"      # e.g. "multisimo"
MODEL = model_id           # e.g. "gpt2-large"
PPLS = ["p1", "p2", "p3"]  # which folders to scan

data, out_data = compute_dominance(DATASET, MODEL, PPLS)

In [30]:
cols = ["speaker_1_dom_score", "speaker_2_dom_score", 
        "speaker_1_1", "speaker_1_2", "speaker_1_3", "speaker_1_4", "speaker_1_5", "speaker_2_1", "speaker_2_2", "speaker_2_3", "speaker_2_4", "speaker_2_5", 
        "<SPK1>_p1", "<SPK2>_p1", "<SPK1>_p2", "<SPK2>_p2", "<SPK1>_p3", "<SPK2>_p3"]
comparison_ppl_dom_data = data.merge(multisimo_df_with_annotations, on="file_name")[cols]
comparison_ppl_dom_data

,speaker_1_dom_score,speaker_2_dom_score,speaker_1_1,speaker_1_2,speaker_1_3,speaker_1_4,speaker_1_5,speaker_2_1,speaker_2_2,speaker_2_3,speaker_2_4,speaker_2_5,<SPK1>_p1,<SPK2>_p1,<SPK1>_p2,<SPK2>_p2,<SPK1>_p3,<SPK2>_p3
0,2.0,3.4,3.0,2.0,2.0,2.0,1.0,4.0,3.0,3.0,4.0,3.0,3.859673,3.692212,2.094028,2.086009,3.204166,2.950220
1,2.0,3.4,2.0,2.0,2.0,2.0,2.0,4.0,3.0,3.0,5.0,2.0,3.614844,3.973222,1.877160,1.985627,3.014114,2.993320
2,2.0,1.6,2.0,1.0,2.0,4.0,1.0,1.0,1.0,1.0,4.0,1.0,3.565423,3.833445,1.904329,2.100131,3.788779,3.233173
3,3.8,2.0,4.0,4.0,4.0,4.0,3.0,3.0,2.0,3.0,1.0,1.0,3.833553,3.412965,2.005451,2.161347,2.988865,3.126620
4,3.4,3.4,4.0,2.0,3.0,4.0,4.0,4.0,2.0,3.0,5.0,3.0,3.694508,3.336642,1.812375,1.622964,3.219100,3.290029
5,2.4,2.8,4.0,2.0,3.0,2.0,1.0,3.0,3.0,3.0,3.0,2.0,3.595748,3.686150,1.761660,1.737376,3.569559,3.357007
6,3.6,3.6,4.0,3.0,3.0,5.0,3.0,4.0,3.0,3.0,5.0,3.0,3.457629,3.321281,1.845813,1.785329,2.989340,3.039983
7,2.8,2.6,3.0,2.0,3.0,4.0,2.0,3.0,2.0,2.0,4.0,2.0,3.699754,3.808591,1.995611,1.919434,3.582191,3.303069
8,3.6,1.4,5.0,2.0,3.0,3.0,5.0,2.0,1.0,2.0,1.0,1.0,3.855456,3.361408,2.043912,1.915491,3.512773,3.464343
9,2.4,1.4,3.0,1.0,2.0,4.0,2.0,2.0,1.0,2.0,1.0,1.0,3.480789,3.957331,1.685946,2.065628,3.404395,3.544259


In [47]:
col_1 = ["speaker_1_dom_score", "speaker_2_dom_score"]
col_2 = ["<SPK1>_p1","<SPK1>_p2","<SPK1>_p3","<SPK2>_p1","<SPK2>_p2","<SPK2>_p3"]
#col_1 = ["p1", "p2", "p3"]
#col_2 = ["dom"]

In [48]:
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_2,comparison_ppl_dom_data)

In [49]:
fig_corr

In [50]:
fig_p

In [38]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
df = data.dropna(subset=["dom","p1","p2","p3"]).copy()
sc = StandardScaler()
Z = sc.fit_transform(df[["p1","p2","p3"]])
X = sm.add_constant(pd.DataFrame(Z, columns=["p1","p2","p3"], index=df.index))
y = df["dom"]
m = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": df["file_name"]})
print(m.summary())

KeyError: ['dom', 'p1', 'p2', 'p3']

### Analysis

In [ ]:
name="S18"
dialog = multisimo_df_with_annotations[multisimo_df_with_annotations['file_name']==file_name]["file_content"][12]

In [ ]:
all_data = pd.DataFrame({})
for p_name in ['p1','p2','p3']:
    file_name_sample = f'/u/sebono/conversational_dominance/notebooks/information_exchange_labelling/dataset_perplexity_results/multisimo_{p_name}_{model_id}/dominance_scores_{name}.pkl'
    with open(file_name_sample, 'rb') as f:
        all_data[f"{p_name}"] = pickle.load(f)[name]

In [ ]:
pattern = r'<(?:SPK[0-9]|MOD)>'
dialog_lines = re.sub(r"[\[\(].*?[\]\)]", "", dialog).replace("<", "\n<").split("\n")[1:]
matches = [f"{match}" for match in re.findall(pattern, f"{start_of_sentence}".join(dialog_lines))]
token_list = [tokenizer(token, return_tensors="pt").input_ids[0] for token in dialog_lines]
encodings = tokenizer(f"{start_of_sentence}".join(dialog_lines), return_tensors="pt")
assert np.cumsum([len(token) for token in token_list])[-1] == encodings.input_ids[0].shape
assert len(matches) == len(dialog_lines)

In [ ]:
all_data.bfill(inplace=True)

filtered_tokens_p1, filtered_ppl_p1, filtered_encodings_p1, filtered_matches_p1 = remove_match_prefix_ppl(token_list, all_data['p1'], matches, tokenizer, min_len=10)
assert np.cumsum([len(token) for token in filtered_tokens_p1])[-1], len(filtered_ppl)

filtered_tokens_p2, filtered_ppl_p2, filtered_encodings_p2, filtered_matches_p2 = remove_match_prefix_ppl(token_list, all_data['p2'], matches, tokenizer, min_len=10)
assert np.cumsum([len(token) for token in filtered_tokens_p2])[-1], len(filtered_ppl)

filtered_tokens_p3, filtered_ppl_p3, filtered_encodings_p3, filtered_matches_p3 = remove_match_prefix_ppl(token_list, all_data['p3'], matches, tokenizer, min_len=10)
assert np.cumsum([len(token) for token in filtered_tokens_p3])[-1], len(filtered_ppl)

assert len(filtered_tokens_p1) == len(filtered_tokens_p2)
assert len(filtered_tokens_p1) == len(filtered_tokens_p3)
assert filtered_encodings_p1 == filtered_encodings_p2
assert filtered_encodings_p1 == filtered_encodings_p3
filtered_encodings = filtered_encodings_p1

all_data_filtered = pd.DataFrame({"p1":filtered_ppl_p1,"p2":filtered_ppl_p2, "p3":filtered_ppl_p3 })

In [ ]:
last=50
end = np.cumsum([len(l) for l in filtered_tokens_p1[:last]])[-1]
compute_graph_perplexity(tokenizer, filtered_tokens_p2[:last], filtered_ppl_p2[:end], filtered_ppl_p3[:end], filtered_matches_p2[:last], {"p1":"p2","p2":"p3"}, answers=None)

In [ ]:
ppls_p1_spk = compute_dominance_per_spk(filtered_ppl_p1, filtered_tokens_p1, filtered_matches_p1, tokenizer)
ppls_p2_spk = compute_dominance_per_spk(filtered_ppl_p2, filtered_tokens_p2, filtered_matches_p2, tokenizer)
ppls_p3_spk = compute_dominance_per_spk(filtered_ppl_p3, filtered_tokens_p3, filtered_matches_p3, tokenizer)

In [ ]:
np.mean(ppls_p2_spk["<SPK1>"])

In [ ]:
np.mean(ppls_p2_spk["<SPK2>"])

In [ ]:
np.mean(ppls_p3_spk['<SPK1>']), np.mean(ppls_p3_spk['<SPK2>'])

In [ ]:
np.mean(ppls_p1_spk['<SPK1>']) > np.mean(ppls_p1_spk['<SPK2>']), np.mean(ppls_p2_spk['<SPK1>']) > np.mean(ppls_p2_spk['<SPK2>']), np.mean(ppls_p3_spk['<SPK1>']) < np.mean(ppls_p3_spk['<SPK2>'])

In [ ]:
rows, nested = compute_significance(ppls_p2_spk, ['<SPK1>', '<SPK2>'], adjust_p="fdr_bh")

# If you want a DataFrame:
import pandas as pd
df = pd.DataFrame(rows).sort_values(["test","p_value"])
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.kdeplot(ppls_p1_spk[f"<SPK1>"], label="spk0", fill=True)
sns.kdeplot(ppls_p1_spk[f"<SPK2>"], label="spk1", fill=True)
plt.title("PPL Distributions by Speaker")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.kdeplot(ppls_p2_spk[f"<SPK1>"], label="spk0", fill=True)
sns.kdeplot(ppls_p2_spk[f"<SPK2>"], label="spk1", fill=True)
plt.title("PPL Distributions by Speaker")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.kdeplot(ppls_p3_spk[f"<SPK1>"], label="spk0", fill=True)
sns.kdeplot(ppls_p3_spk[f"<SPK2>"], label="spk1", fill=True)
plt.title("PPL Distributions by Speaker")
plt.legend()
plt.show()

In [ ]:
# BEFORE
col_1 = ["p1","p2","p3"]
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_1,all_data_filtered)

In [ ]:
fig_corr

In [ ]:
fig_p

### Per-turn Analysis

In [ ]:
per_turn_ppl = pd.DataFrame({"tokens": filtered_tokens_p1})

In [ ]:
from itertools import count

# Start from offset
start_index = 0
counter = count(start=start_index)

# Function to assign token indices
def assign_indices(token_list):
    return [next(counter) for _ in token_list]

per_turn_ppl["ppl"] = per_turn_ppl["tokens"].apply(assign_indices)

In [ ]:
per_turn_ppl["ppl"].iloc[-1][-1], len(filtered_ppl_p3)

In [ ]:
np.mean(filtered_ppl_p3[0:8])

In [ ]:
per_turn_ppl["speaker"] = filtered_matches_p1

In [ ]:
per_turn_ppl["p1"] = per_turn_ppl["ppl"].apply(
    lambda idxs: np.asarray(filtered_ppl_p1)[idxs].mean(axis=0) if len(idxs) > 0 else np.nan
)
per_turn_ppl["p2"] = per_turn_ppl["ppl"].apply(
    lambda idxs: np.asarray(filtered_ppl_p2)[idxs].mean(axis=0) if len(idxs) > 0 else np.nan
)
per_turn_ppl["p3"] = per_turn_ppl["ppl"].apply(
    lambda idxs: np.asarray(filtered_ppl_p3)[idxs].mean(axis=0) if len(idxs) > 0 else np.nan
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.kdeplot(per_turn_ppl[per_turn_ppl["speaker"]=="<SPK1>"]['p1'], label="spk1", fill=True)
sns.kdeplot(per_turn_ppl[per_turn_ppl["speaker"]=="<SPK2>"]['p1'], label="spk2", fill=True)
plt.title("PPL Distributions by Speaker")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.kdeplot(per_turn_ppl[per_turn_ppl["speaker"]=="<SPK1>"]['p2'], label="spk1", fill=True)
sns.kdeplot(per_turn_ppl[per_turn_ppl["speaker"]=="<SPK2>"]['p2'], label="spk2", fill=True)
plt.title("PPL Distributions by Speaker")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.kdeplot(per_turn_ppl[per_turn_ppl["speaker"]=="<SPK1>"]['p3'], label="spk1", fill=True)
sns.kdeplot(per_turn_ppl[per_turn_ppl["speaker"]=="<SPK2>"]['p3'], label="spk2", fill=True)
plt.title("PPL Distributions by Speaker")
plt.legend()
plt.show()

In [ ]:
ppls_p1_spk = dict({"<SPK1>": per_turn_ppl[per_turn_ppl["speaker"]=="<SPK1>"]['p1'], "<SPK2>": per_turn_ppl[per_turn_ppl["speaker"]=="<SPK2>"]['p1'], "<MOD>":per_turn_ppl[per_turn_ppl["speaker"]=="<MOD>"]['p1']})
ppls_p2_spk = dict({"<SPK1>": per_turn_ppl[per_turn_ppl["speaker"]=="<SPK1>"]['p2'], "<SPK2>": per_turn_ppl[per_turn_ppl["speaker"]=="<SPK2>"]['p2'], "<MOD>":per_turn_ppl[per_turn_ppl["speaker"]=="<MOD>"]['p2']})
ppls_p3_spk = dict({"<SPK1>": per_turn_ppl[per_turn_ppl["speaker"]=="<SPK1>"]['p3'], "<SPK2>": per_turn_ppl[per_turn_ppl["speaker"]=="<SPK2>"]['p3'], "<MOD>":per_turn_ppl[per_turn_ppl["speaker"]=="<MOD>"]['p3']})

In [ ]:
rows, nested = compute_significance(ppls_p1_spk, list(np.unique(filtered_matches_p1)), adjust_p="fdr_bh")

# If you want a DataFrame:
import pandas as pd
df = pd.DataFrame(rows).sort_values(["test","p_value"])
df

In [ ]:
rows, nested = compute_significance(ppls_p2_spk, list(np.unique(filtered_matches_p1)), adjust_p="fdr_bh")

# If you want a DataFrame:
import pandas as pd
df = pd.DataFrame(rows).sort_values(["test","p_value"])
df.head()

In [ ]:
rows, nested = compute_significance(ppls_p3_spk, list(np.unique(filtered_matches_p1)), adjust_p="fdr_bh")

# If you want a DataFrame:
import pandas as pd
df = pd.DataFrame(rows).sort_values(["test","p_value"])
df

In [ ]:
per_turn_ppl["dialog"] = per_turn_ppl["tokens"].map(lambda x: " ".join([tokenizer.decode(token, skip_special_tokens=True) for token in x]))

In [ ]:
# BEFORE
col_1 = ["p1","p2","p3"]
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_1,per_turn_ppl)

In [ ]:
fig_corr

In [ ]:
fig_p